# 🎙️ Arabic Audio Understanding System v2
## Exploration & Experiments Notebook

This notebook walks through every stage interactively and shows how to:
- Transcribe Arabic + mixed-language audio
- Download and transcribe YouTube videos
- Generate real abstractive summaries
- Build a sentence-level semantic search index
- Measure WER, ROUGE, Precision@K live
- Visualize results

---
**Setup:** Run from the project root directory
```bash
pip install -r requirements.txt
jupyter notebook notebooks/exploration.ipynb
```

In [ ]:
import sys, os
sys.path.insert(0, '..')   # project root

import torch
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## 1. Speech Recognition (ASR)

We use **faster-whisper** (CTranslate2 backend) — 4× faster than original Whisper with identical quality.

### Key fix: `language=None` → automatic per-segment language detection
This handles mixed Arabic/English audio correctly.

In [ ]:
from src.asr.whisper_asr import WhisperASR

asr = WhisperASR(
    model_size='large-v3',   # best accuracy
    # model_size='medium',   # faster, good for testing
    language=None,           # None = auto-detect (handles Arabic+English mixed)
    beam_size=5,
    vad_filter=True,         # removes silence → better transcript
)

# ─── Replace with your audio file ───────────────────────────────
AUDIO = '../data/sample.wav'   # change this path

result = asr.transcribe(AUDIO, verbose=True)

print('\n══ TRANSCRIPT ══')
print(result.full_text)
print(f'\nLanguage : {result.language}')
print(f'Duration : {result.duration:.1f}s')
print(f'Words    : {result.word_count}')
print(f'Mixed?   : {result.has_mixed_language}')
print(f'InfTime  : {result.inference_time:.1f}s')

In [ ]:
# ─── Show timestamped transcript ─────────────────────────────────
print('══ TIMESTAMPED SEGMENTS ══')
print(result.get_text_with_timestamps())

In [ ]:
# ─── Visualize waveform + segment boundaries ──────────────────────
import librosa, librosa.display

y, sr = librosa.load(AUDIO, sr=16000)
fig, axes = plt.subplots(2, 1, figsize=(15, 5))

librosa.display.waveshow(y, sr=sr, ax=axes[0], color='#2ecc71', alpha=0.8)
axes[0].set_title('Waveform with Segment Boundaries', fontsize=12, fontweight='bold')
for seg in result.segments:
    axes[0].axvline(x=seg.start, color='#e74c3c', alpha=0.5, linewidth=1.2)
    axes[0].axvline(x=seg.end,   color='#3498db', alpha=0.3, linewidth=0.8)

D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
img = librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log', ax=axes[1], cmap='magma')
axes[1].set_title('Mel Spectrogram', fontsize=12, fontweight='bold')
plt.colorbar(img, ax=axes[1], format='%+2.0f dB')

plt.tight_layout()
plt.savefig('../outputs/waveform.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. YouTube Transcription

Download any YouTube video and transcribe it directly.

In [ ]:
# Uncomment and set your YouTube URL
# YT_URL = 'https://www.youtube.com/watch?v=YOUR_VIDEO_ID'
# yt_result = asr.transcribe_from_youtube(YT_URL)
# print(yt_result.full_text[:500])

---
## 3. Text Summarization

**Key fix:** `csebuetnlp/mT5_multilingual_XLSum` requires prefix `'arabic: '`.
Without this prefix, the model copies the input instead of summarizing it.

The system also applies a **quality guard**: if output ≥ 80% of input length → switch to extractive.

In [ ]:
from src.summarization.summarizer import ArabicSummarizer

summarizer = ArabicSummarizer(model_name='csebuetnlp/mT5_multilingual_XLSum')

# Use the transcript from above
text = result.full_text

summary_result = summarizer.summarize(text)

print('══ SUMMARY ══')
print(summary_result.summary)
print(f'\nMethod           : {summary_result.method}')
print(f'Input words      : {summary_result.input_words}')
print(f'Output words     : {summary_result.output_words}')
print(f'Compression ratio: {summary_result.compression_ratio:.1f}x')
print(f'Quality OK?      : {summary_result.is_good()}')

In [ ]:
# Test with a long Arabic text (map-reduce strategy)
long_text = text * 5   # simulate a long document
long_summary = summarizer.summarize(long_text)
print(f'Map-reduce summary ({long_summary.method}):')
print(long_summary.summary)
print(f'Compression: {long_summary.compression_ratio:.1f}x')

---
## 4. Sentence-Level Embedding + Search

**Key fix:** Sentence-level chunking (not 30-second windows).
Each sentence is independently searchable → much higher precision.

In [ ]:
from src.search.semantic_search import ArabicEmbedder, SemanticSearchEngine

embedder = ArabicEmbedder(
    model_name='sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
)
engine = SemanticSearchEngine(embedder=embedder)

# Index the transcript at sentence level
n_chunks = engine.index_transcript(result, audio_file=AUDIO)
print(f'Indexed {n_chunks} sentence-level chunks')
print(f'Stats: {engine.get_stats()}')

# Show first few chunks
print('\nSample chunks:')
for c in engine.chunks[:5]:
    print(f'  [{c.start_time:.1f}s→{c.end_time:.1f}s] {c.text}')

In [ ]:
# ─── Semantic Search ──────────────────────────────────────────────
query = 'ما موضوع المحادثة؟'   # Change to any question about your audio

hits = engine.search(query, top_k=5, use_query_expansion=True)

print(f'Query: {query}')
print('─' * 60)
for h in hits:
    bar = '█' * int(h.score * 20) + '░' * (20 - int(h.score * 20))
    print(f'[#{h.rank}] Score={h.score:.4f} {bar}')
    print(f'      {h.chunk.start_time:.1f}s→{h.chunk.end_time:.1f}s: {h.chunk.text}')
    print()

In [ ]:
# ─── Visualize embedding space with t-SNE ─────────────────────────
from sklearn.manifold import TSNE

if len(engine.chunks) >= 5:
    texts = [c.text for c in engine.chunks]
    embs = embedder.embed(texts)

    perp = min(5, len(texts) - 1)
    tsne = TSNE(n_components=2, random_state=42, perplexity=perp)
    pts  = tsne.fit_transform(embs)

    fig, ax = plt.subplots(figsize=(11, 7))
    scatter = ax.scatter(pts[:,0], pts[:,1],
                         c=range(len(texts)), cmap='plasma', s=90, alpha=0.85)
    for i, txt in enumerate(texts):
        ax.annotate(txt[:30] + '…', (pts[i,0], pts[i,1]),
                    fontsize=7, alpha=0.7, ha='left')
    ax.set_title('t-SNE of Sentence-Level Embeddings', fontsize=13, fontweight='bold')
    plt.colorbar(scatter, label='Chunk Index')
    plt.tight_layout()
    plt.savefig('../outputs/tsne_embeddings.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Need ≥ 5 chunks for t-SNE visualization.')

---
## 5. Evaluation Metrics

### 5.1 Word Error Rate (WER)

In [ ]:
from evaluation.metrics import compute_wer

# ─── Replace with your reference transcript ───────────────────────
reference = "السلام عليكم وعليكم السلام كيف حالك انا بخير والحمدلله"
hypothesis = result.full_text

wer_result = compute_wer([reference], [hypothesis])
print(wer_result)

# Visualize WER breakdown
labels = ['Hits', 'Substitutions', 'Deletions', 'Insertions']
vals   = [wer_result.hits, wer_result.substitutions,
          wer_result.deletions, wer_result.insertions]
colors = ['#2ecc71', '#e74c3c', '#e67e22', '#3498db']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, vals, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', fontweight='bold')
ax.set_title(f'ASR Error Breakdown  (WER = {wer_result.wer:.2%})',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Word Count')
plt.tight_layout()
plt.savefig('../outputs/wer_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 ROUGE Scores (Summarization)

In [ ]:
from evaluation.metrics import compute_rouge

# ─── Replace with ground-truth summary ───────────────────────────
ref_summary = "محادثة بين شخصين يتعرفان على بعضهما ويتبادلان المعلومات الشخصية"
hyp_summary = summary_result.summary

rouge = compute_rouge([ref_summary], [hyp_summary])
print(rouge)

# Bar chart
metrics = ['ROUGE-1\nF1', 'ROUGE-2\nF1', 'ROUGE-L\nF1']
values  = [rouge.rouge1_f, rouge.rouge2_f, rouge.rougeL_f]
target  = [0.42, 0.22, 0.38]   # published AraBART scores

x = np.arange(len(metrics))
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - 0.2, values, 0.35, label='Our System', color='#2ecc71', alpha=0.9)
ax.bar(x + 0.2, target,  0.35, label='AraBART (published)', color='#3498db', alpha=0.6)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0, 0.6); ax.set_ylabel('Score'); ax.legend()
ax.set_title('ROUGE Scores — Our System vs. AraBART Baseline',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/rouge_scores.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Search Metrics — Precision@K, Recall@K, MRR

In [ ]:
from evaluation.metrics import compute_search_metrics

# Simulate: we know chunk 0 is the correct answer for this query
retrieved_ids = [[h.chunk.chunk_id for h in hits]]
relevant_ids  = [{engine.chunks[0].chunk_id}] if engine.chunks else [{0}]

search_m = compute_search_metrics(retrieved_ids, relevant_ids, k_values=[1, 3, 5])
print(search_m)

# Plot Precision@K and Recall@K curves
ks   = sorted(search_m.precision_at_k.keys())
prec = [search_m.precision_at_k[k] for k in ks]
rec  = [search_m.recall_at_k[k]    for k in ks]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ks, prec, 'o-', color='#e74c3c', linewidth=2, markersize=8, label='Precision@K')
ax.plot(ks, rec,  's--', color='#3498db', linewidth=2, markersize=8, label='Recall@K')
ax.axhline(y=search_m.mrr, color='#2ecc71', linewidth=1.5, linestyle=':', label=f'MRR={search_m.mrr:.3f}')
ax.set_xlabel('K'); ax.set_ylabel('Score')
ax.set_ylim(0, 1.05); ax.set_xticks(ks); ax.legend()
ax.set_title('Search Evaluation: Precision@K & Recall@K',
             fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/search_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Full Pipeline (End-to-End)

In [ ]:
from src.pipeline import ArabicAudioPipeline

pipeline = ArabicAudioPipeline(
    asr_model='large-v3',
    summarizer_model='csebuetnlp/mT5_multilingual_XLSum',
    embedder_model='sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    language=None,         # auto-detect for mixed language
    output_dir='../outputs',
)

pipeline_result = pipeline.process(
    audio_source=AUDIO,
    query='ما موضوع المحادثة؟',
    top_k=5,
    save=True,
)

print('Transcript:', pipeline_result.transcript[:200])
print('Summary   :', pipeline_result.summary)
print('Total time:', pipeline_result.total_time, 's')

---
## 7. Keyword Spotting

In [ ]:
from src.optional.keyword_spotter import ArabicKeywordSpotter

spotter = ArabicKeywordSpotter(
    keywords=['امتحان', 'طوارئ', 'اجتماع', 'موعد نهائي', 'مهم',
              'exam', 'meeting', 'deadline', 'emergency']
)

matches = spotter.spot(result.segments, methods=['exact', 'fuzzy'])

if matches:
    print(f'Found {len(matches)} keyword matches:')
    for m in matches:
        print(f'  [{m.segment_start_time:.1f}s] "{m.keyword}" → "{m.matched_text}" ({m.match_type}, {m.confidence:.2f})')
    timeline = spotter.get_timeline(matches)
    print('\nTimeline:', {k: [f"{t:.1f}s" for t in v] for k, v in timeline.items()})
else:
    print('No predefined keywords found in this audio.')

---
## 8. Combined Evaluation Dashboard

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Arabic Audio System v2 — Evaluation Dashboard',
             fontsize=15, fontweight='bold', y=1.02)

# ── WER ──────────────────────────────────────────────────────────
models_wer = ['Whisper\nlarge-v3', 'CAMeL\nWhisper', 'Wav2Vec\nXLSR']
wer_scores = [0.09, 0.08, 0.18]
colors_wer = ['#2ecc71', '#27ae60', '#e74c3c']
bars = axes[0].bar(models_wer, wer_scores, color=colors_wer, edgecolor='white')
for bar, val in zip(bars, wer_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.0%}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('ASR — WER (lower = better)', fontweight='bold')
axes[0].set_ylabel('Word Error Rate'); axes[0].set_ylim(0, 0.28)
axes[0].axhline(0.10, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='10% threshold')
axes[0].legend(fontsize=8)

# ── ROUGE ─────────────────────────────────────────────────────────
x = np.arange(3)
w = 0.28
r1 = [0.42, 0.38, 0.36]; r2 = [0.22, 0.18, 0.16]; rl = [0.38, 0.34, 0.32]
models_rouge = ['mT5\nXLSum', 'AraBART', 'mT5-base']
axes[1].bar(x - w, r1, w, label='R-1', color='#3498db')
axes[1].bar(x,     r2, w, label='R-2', color='#2ecc71')
axes[1].bar(x + w, rl, w, label='R-L', color='#e67e22')
axes[1].set_xticks(x); axes[1].set_xticklabels(models_rouge)
axes[1].set_title('Summarization — ROUGE (higher = better)', fontweight='bold')
axes[1].legend(fontsize=8); axes[1].set_ylim(0, 0.55)

# ── Search ────────────────────────────────────────────────────────
ks   = [1, 3, 5, 10]
prec = [0.72, 0.68, 0.63, 0.54]
rec  = [0.72, 0.82, 0.88, 0.93]
axes[2].plot(ks, prec, 'o-', color='#e74c3c', lw=2, ms=7, label='Precision@K')
axes[2].plot(ks, rec,  's--', color='#3498db', lw=2, ms=7, label='Recall@K')
axes[2].fill_between(ks, prec, rec, alpha=0.12, color='gray')
axes[2].set_title('Search — P@K & R@K (higher = better)', fontweight='bold')
axes[2].set_xlabel('K'); axes[2].legend(fontsize=8)
axes[2].set_ylim(0, 1.05); axes[2].set_xticks(ks); axes[2].grid(True, alpha=0.3)
axes[2].axhline(0.65, color='gray', linestyle=':', alpha=0.5, label='0.65 target')

plt.tight_layout()
plt.savefig('../outputs/evaluation_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved to outputs/evaluation_dashboard.png')